In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv("../../data/staging/stg_walmart_demand.csv")
forecast = pd.read_csv("../../data/processed/demand_forecast.csv")

df.head()

,store_id,department_id,date,weekly_sales_amount,isholiday,temperature,fuel_price,markdown1,markdown2,markdown3,markdown4,markdown5,cpi,unemployment,store_type,store_size,sku_id,location_id,demand_units
0,1,1,2010-02-05,24924.50,False,42.31,2.572,NaN,NaN,NaN,NaN,NaN,211.096358,8.106,A,151315,SKU_001,STORE_001,249
1,1,1,2010-02-12,46039.49,True,38.51,2.548,NaN,NaN,NaN,NaN,NaN,211.242170,8.106,A,151315,SKU_001,STORE_001,460
2,1,1,2010-02-19,41595.55,False,39.93,2.514,NaN,NaN,NaN,NaN,NaN,211.289143,8.106,A,151315,SKU_001,STORE_001,416
3,1,1,2010-02-26,19403.54,False,46.63,2.561,NaN,NaN,NaN,NaN,NaN,211.319643,8.106,A,151315,SKU_001,STORE_001,194
4,1,1,2010-03-05,21827.90,False,46.50,2.625,NaN,NaN,NaN,NaN,NaN,211.350143,8.106,A,151315,SKU_001,STORE_001,218


In [2]:
sku_stats = df.groupby("sku_id")["demand_units"].agg(
    ["mean", "std"]
).reset_index()

sku_stats.columns = [
    "sku_id",
    "avg_demand",
    "demand_std"
]

sku_stats.head()

,sku_id,avg_demand,demand_std
0,SKU_001,192.133800,151.022843
1,SKU_002,436.078477,251.762959
2,SKU_003,117.940016,127.903546
3,SKU_004,259.745144,132.611853
4,SKU_005,213.658736,199.885268


In [3]:
sku_sales = (
    df.groupby("sku_id")["demand_units"]
    .sum()
    .sort_values(ascending=False)
)

sku_sales = sku_sales.reset_index()
sku_sales["cum_pct"] = (
    sku_sales["demand_units"].cumsum()
    / sku_sales["demand_units"].sum()
)

def classify_abc(x):
    if x <= 0.8:
        return "A"
    elif x <= 0.95:
        return "B"
    return "C"

sku_sales["abc_class"] = sku_sales["cum_pct"].apply(classify_abc)

sku_sales = sku_sales[["sku_id", "abc_class"]]

sku_sales.head()

,sku_id,abc_class
0,SKU_092,A
1,SKU_095,A
2,SKU_038,A
3,SKU_072,A
4,SKU_090,A


In [4]:
inventory_df = sku_stats.merge(
    sku_sales,
    on="sku_id",
    how="left"
)

inventory_df.head()

,sku_id,avg_demand,demand_std,abc_class
0,SKU_001,192.133800,151.022843,A
1,SKU_002,436.078477,251.762959,A
2,SKU_003,117.940016,127.903546,B
3,SKU_004,259.745144,132.611853,A
4,SKU_005,213.658736,199.885268,A


In [5]:
avg_lead_time = 3.47
lead_time_std = 1.67

In [6]:
z_map = {
    "A": 2.05,
    "B": 1.65,
    "C": 1.28
}

inventory_df["z_score"] = inventory_df["abc_class"].map(z_map)

inventory_df.head()

,sku_id,avg_demand,demand_std,abc_class,z_score
0,SKU_001,192.133800,151.022843,A,2.05
1,SKU_002,436.078477,251.762959,A,2.05
2,SKU_003,117.940016,127.903546,B,1.65
3,SKU_004,259.745144,132.611853,A,2.05
4,SKU_005,213.658736,199.885268,A,2.05


In [7]:
inventory_df["safety_stock"] = (
    inventory_df["z_score"] *
    np.sqrt(
        (avg_lead_time * (inventory_df["demand_std"] ** 2)) +
        ((inventory_df["avg_demand"] ** 2) * (lead_time_std ** 2))
    )
)

inventory_df.head()

,sku_id,avg_demand,demand_std,abc_class,z_score,safety_stock
0,SKU_001,192.133800,151.022843,A,2.05,874.792340
1,SKU_002,436.078477,251.762959,A,2.05,1775.700155
2,SKU_003,117.940016,127.903546,B,1.65,510.060889
3,SKU_004,259.745144,132.611853,A,2.05,1023.324410
4,SKU_005,213.658736,199.885268,A,2.05,1057.200398


In [8]:
inventory_df["reorder_point"] = (
    inventory_df["avg_demand"] * avg_lead_time
) + inventory_df["safety_stock"]

inventory_df.head()

,sku_id,avg_demand,demand_std,abc_class,z_score,safety_stock,reorder_point
0,SKU_001,192.133800,151.022843,A,2.05,874.792340,1541.496624
1,SKU_002,436.078477,251.762959,A,2.05,1775.700155,3288.892470
2,SKU_003,117.940016,127.903546,B,1.65,510.060889,919.312743
3,SKU_004,259.745144,132.611853,A,2.05,1023.324410,1924.640059
4,SKU_005,213.658736,199.885268,A,2.05,1057.200398,1798.596214


In [20]:
inventory_df["target_inventory"] = (
    inventory_df["avg_demand"] * avg_lead_time
) + inventory_df["safety_stock"]

inventory_df.head()

,sku_id,avg_demand,demand_std,abc_class,z_score,safety_stock,reorder_point,target_inventory
0,SKU_001,192.133800,151.022843,A,2.05,874.792340,1541.496624,1541.496624
1,SKU_002,436.078477,251.762959,A,2.05,1775.700155,3288.892470,3288.892470
2,SKU_003,117.940016,127.903546,B,1.65,510.060889,919.312743,919.312743
3,SKU_004,259.745144,132.611853,A,2.05,1023.324410,1924.640059,1924.640059
4,SKU_005,213.658736,199.885268,A,2.05,1057.200398,1798.596214,1798.596214


In [23]:
inventory_policy = inventory_df[
    [
        "sku_id",
        "abc_class",
        "avg_demand",
        "demand_std",
        "safety_stock",
        "reorder_point",
        "target_inventory"
    ]
]

inventory_policy.to_csv(
    "../../data/processed/inventory_policy.csv",
    index=False
)

In [12]:
inventory_policy.to_csv(
    "../../data/processed/inventory_policy.csv",
    index=False
)

inventory_policy.head()

,sku_id,abc_class,avg_demand,demand_std,safety_stock,reorder_point,target_inventory
0,SKU_001,A,192.133800,151.022843,874.792340,1541.496624,458638.029006
1,SKU_002,A,436.078477,251.762959,1775.700155,3288.892470,459538.936821
2,SKU_003,B,117.940016,127.903546,510.060889,919.312743,458273.297555
3,SKU_004,A,259.745144,132.611853,1023.324410,1924.640059,458786.561077
4,SKU_005,A,213.658736,199.885268,1057.200398,1798.596214,458820.437065


In [13]:
inventory_policy.sort_values(
    "safety_stock",
    ascending=False
).head(5)

,sku_id,abc_class,avg_demand,demand_std,safety_stock,reorder_point,target_inventory
73,SKU_092,A,752.048019,494.140266,3192.095145,5801.701770,460955.331812
76,SKU_095,A,698.239627,382.005487,2800.383412,5223.274918,460563.620079
60,SKU_072,A,505.674165,447.110612,2431.493480,4186.182832,460194.730147
36,SKU_038,A,610.905051,239.671414,2282.927396,4402.767921,460046.164062
71,SKU_090,A,452.323077,324.620440,1983.592330,3553.153407,459746.828997


In [14]:
inventory_policy.sort_values(
    "reorder_point",
    ascending=False
).head(5)

,sku_id,abc_class,avg_demand,demand_std,safety_stock,reorder_point,target_inventory
73,SKU_092,A,752.048019,494.140266,3192.095145,5801.701770,460955.331812
76,SKU_095,A,698.239627,382.005487,2800.383412,5223.274918,460563.620079
36,SKU_038,A,610.905051,239.671414,2282.927396,4402.767921,460046.164062
60,SKU_072,A,505.674165,447.110612,2431.493480,4186.182832,460194.730147
71,SKU_090,A,452.323077,324.620440,1983.592330,3553.153407,459746.828997


In [15]:
inventory_policy.isnull().sum()

sku_id              0
abc_class           0
avg_demand          0
demand_std          0
safety_stock        0
reorder_point       0
target_inventory    0
dtype: int64

In [16]:
(inventory_policy < 0).sum(numeric_only=True)

TypeError: '<' not supported between instances of 'str' and 'int'

In [17]:
numeric_cols = inventory_policy.select_dtypes(include="number").columns

(inventory_policy[numeric_cols] < 0).sum()

avg_demand          0
demand_std          0
safety_stock        0
reorder_point       0
target_inventory    0
dtype: int64